# Módulo 9 · Tarea 1: Lectura y exploración de datos

Exploración del archivo `data/tareas.csv`, exportado desde la base de datos de la app con `python scripts/exportar_tareas.py`.

In [1]:
import pandas as pd

RUTA_CSV = "../data/tareas.csv"

df = pd.read_csv(RUTA_CSV)
df.head()

,id,titulo,estado,fecha_creacion,categoria,fecha_limite,prioridad,tiempo_estimado,completado_en
0,1,Estudiar SQLite,completada,2026-09-09 23:06:31,General,NaN,media,NaN,NaN
1,4,practicar formularios,pendiente,2026-09-29 18:51:15,Estudio,NaN,media,NaN,NaN
2,5,Lavar ropa,completada,2026-09-01 09:00:00,Hogar,2026-09-02 20:00:00,media,60.0,2026-09-01 10:20:00
3,6,Hacer el súper,completada,2026-09-03 10:00:00,compras,2026-09-04 20:00:00,Alta,90.0,2026-09-03 11:45:00
4,7,Pagar la luz,completada,2026-09-05 08:00:00,Personal,2026-09-08 18:00:00,alta,15.0,2026-09-10 09:00:00


## 1. Inspección de columnas

El modelo `Tarea` extendido guarda los campos que pide la tarea con estos nombres:

| Campo de la tarea | Columna en la base |
|---|---|
| `created_at` | `fecha_creacion` |
| `due_date` | `fecha_limite` |
| `priority` | `prioridad` |
| `estimated_time` | `tiempo_estimado` (minutos) |
| `completed_at` | `completado_en` |

In [2]:
print("Filas y columnas:", df.shape)
print("Columnas:", list(df.columns))
df.info()

Filas y columnas: (10, 9)
Columnas: ['id', 'titulo', 'estado', 'fecha_creacion', 'categoria', 'fecha_limite', 'prioridad', 'tiempo_estimado', 'completado_en']
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               10 non-null     int64  
 1   titulo           10 non-null     str    
 2   estado           10 non-null     str    
 3   fecha_creacion   10 non-null     str    
 4   categoria        10 non-null     str    
 5   fecha_limite     8 non-null      str    
 6   prioridad        10 non-null     str    
 7   tiempo_estimado  7 non-null      float64
 8   completado_en    6 non-null      str    
dtypes: float64(1), int64(1), str(7)
memory usage: 852.0 bytes


In [3]:
CAMPOS_EXTENDIDOS = ["fecha_creacion", "fecha_limite", "prioridad", "tiempo_estimado", "completado_en"]

faltantes = [c for c in CAMPOS_EXTENDIDOS if c not in df.columns]
print("Campos extendidos presentes:", not faltantes)

df[CAMPOS_EXTENDIDOS].describe(include="all")

Campos extendidos presentes: True


,fecha_creacion,fecha_limite,prioridad,tiempo_estimado,completado_en
count,10,8,10,7.000000,6
unique,10,8,6,NaN,6
top,2026-09-09 23:06:31,2026-09-02 20:00:00,media,NaN,2026-09-01 10:20:00
freq,1,1,4,NaN,1
mean,NaN,NaN,NaN,35.714286,NaN
std,NaN,NaN,NaN,37.464269,NaN
min,NaN,NaN,NaN,-30.000000,NaN
25%,NaN,NaN,NaN,22.500000,NaN
50%,NaN,NaN,NaN,40.000000,NaN
75%,NaN,NaN,NaN,52.500000,NaN


In [4]:
for columna in ["estado", "categoria", "prioridad"]:
    print(f"--- {columna} ---")
    print(df[columna].value_counts(dropna=False).to_string(), end="\n\n")

--- estado ---
estado
completada    7
pendiente     3

--- categoria ---
categoria
Hogar        2
Personal     2
General      1
Estudio      1
compras      1
Ejercicio    1
HOGAR        1
Mascotas     1

--- prioridad ---
prioridad
media      4
baja       2
Alta       1
alta       1
urgente    1
 BAJA      1



Se observan valores inconsistentes (por ejemplo `Hogar` y `HOGAR `, o `Alta`, ` BAJA` y `urgente`). Se limpian en `m09-analisis.ipynb`.

## 2. Filtrar y ordenar por dos columnas diferentes

Filtro por `estado` (completadas) y `tiempo_estimado` (45 minutos o más); orden por `categoria` (A→Z) y `tiempo_estimado` (mayor a menor).

In [5]:
completadas_largas = df[(df["estado"] == "completada") & (df["tiempo_estimado"] >= 45)]

completadas_largas.sort_values(by=["categoria", "tiempo_estimado"], ascending=[True, False])[
    ["id", "titulo", "categoria", "prioridad", "tiempo_estimado", "fecha_limite"]
]

,id,titulo,categoria,prioridad,tiempo_estimado,fecha_limite
2,5,Lavar ropa,Hogar,media,60.0,2026-09-02 20:00:00
5,8,Hacer la comida,Hogar,urgente,45.0,2026-09-08 15:00:00
3,6,Hacer el súper,compras,Alta,90.0,2026-09-04 20:00:00


Otro ejemplo: tareas de prioridad alta ordenadas por `fecha_limite` y `fecha_creacion`.

In [6]:
prioridad_normalizada = df["prioridad"].str.strip().str.lower()

df[prioridad_normalizada == "alta"].sort_values(by=["fecha_limite", "fecha_creacion"])[
    ["id", "titulo", "prioridad", "fecha_limite", "fecha_creacion", "estado"]
]

,id,titulo,prioridad,fecha_limite,fecha_creacion,estado
3,6,Hacer el súper,Alta,2026-09-04 20:00:00,2026-09-03 10:00:00,completada
4,7,Pagar la luz,alta,2026-09-08 18:00:00,2026-09-05 08:00:00,completada


## 3. Agrupar tareas por categoría

Primero con los datos tal como vienen y después normalizando el texto de la categoría.

In [7]:
df.groupby("categoria")["id"].count().rename("total_tareas")

categoria
Ejercicio    1
Estudio      1
General      1
HOGAR        1
Hogar        2
Mascotas     1
Personal     2
compras      1
Name: total_tareas, dtype: int64

In [8]:
df["categoria_normalizada"] = df["categoria"].str.strip().str.capitalize()

resumen_categoria = df.groupby("categoria_normalizada").agg(
    total_tareas=("id", "count"),
    completadas=("estado", lambda s: (s == "completada").sum()),
    tiempo_estimado_promedio=("tiempo_estimado", "mean"),
).sort_values("total_tareas", ascending=False)

resumen_categoria.round(1)

,total_tareas,completadas,tiempo_estimado_promedio
categoria_normalizada,,,
Hogar,3,3,25.0
Personal,2,1,15.0
Compras,1,1,90.0
Estudio,1,0,NaN
Ejercicio,1,1,40.0
General,1,1,NaN
Mascotas,1,0,30.0


## 4. Estadísticas básicas

In [9]:
total_tareas = len(df)
promedio_estimado = df["tiempo_estimado"].mean()
porcentaje_completadas = (df["estado"] == "completada").mean() * 100

print(f"Número total de tareas:          {total_tareas}")
print(f"Promedio de tiempo estimado:     {promedio_estimado:.1f} minutos")
print(f"Porcentaje de tareas completadas: {porcentaje_completadas:.1f} %")

Número total de tareas:          10
Promedio de tiempo estimado:     35.7 minutos
Porcentaje de tareas completadas: 70.0 %


El promedio de `tiempo_estimado` todavía incluye un valor negativo y omite los vacíos; en el notebook de análisis se corrige antes de sacar conclusiones.